# 🚀 AI Video Dubbing Studio v2 - Colab + Kaggle, Dual-T4 Optimized

This is the **v2** notebook: it auto-detects whether it's running on **Google Colab** or **Kaggle**, and on Kaggle with 2 GPUs selected, it automatically **splits voice synthesis (the slowest stage) across both T4 GPUs in parallel** for a real speedup. Everything else (Colab, or Kaggle with 1 GPU) runs the same proven single-GPU pipeline as before.

### 📋 Setup:
1. **Colab**: Runtime > Change runtime type > **T4 GPU** (or A100).
   **Kaggle**: Notebook Settings (right sidebar) > Accelerator > **GPU T4 x2** for the dual-GPU speedup (a single T4 also works, it just won't parallelize). Kaggle also needs **Internet: On** in the same settings panel, or Cell 1's git clone/pip installs will fail.
2. Run **Cell 1** (detects your environment, installs dependencies including `ipywidgets`, downloads model weights).

### Then pick ONE of these two ways to dub a video:

**Option A - Everything inside the notebook, no local app needed (easiest):**
- Run **Cell 3** below - it displays an interactive form (dropdowns, text boxes, a Start button) that looks and works the same on both Colab and Kaggle. Fill it in, click **Start Processing**. On Kaggle with 2 GPUs, you'll see `[GPU0]` and `[GPU1]` tagged log lines running side by side during voice synthesis.

**Option B - Use the local Video Dubbing GUI on your own PC:**
- Run **Cell 2** (Start Cloud GPU Server & Tunnel) instead of Cell 3.
- Copy the public **Cloudflare Tunnel URL** it prints and paste it into your local **Video Dubbing GUI**'s "Remote Cloud GPU" field. (Dual-GPU sharding is Cell 3-only for now - Option B always uses a single GPU.)

### 🔁 Crash-safe by design:
Every stage's progress is checkpointed to disk. If a run gets killed partway through (e.g. an out-of-memory kill during a long video), just **re-run Cell 3 with the same settings** - it automatically resumes from the last completed stage/segment instead of starting over. This applies on both the single-GPU and dual-GPU paths.

### 🔑 Groq API Key (translation) - 1 required + 4 optional fallback keys:
Cell 2 and Cell 3 both have a `groq_api_key` field (required for AI-powered context-aware translation) plus `groq_api_key_2` through `groq_api_key_5` (optional). If the primary key hits Groq's free-tier rate limit partway through a long video, processing automatically shifts to the next key, then the next. Leaving all of them blank falls back to the local translator (no AI condensing).

### 📁 Video sources:
- **Upload from computer** - Colab only (Kaggle notebooks don't support in-cell file upload widgets).
- **YouTube URL** - works on both. If it fails, Colab/Kaggle's IP may be bot-blocked by YouTube; export a `cookies.txt` from a logged-in browser session and paste its path into the **YouTube Cookies File** field.
- **Google Drive Link** - works on both, and is the recommended option for Kaggle. Paste the file's **Share** link and make sure it's shared as **"Anyone with the link"**.

### ⚙️ How the dual-GPU split works:
On Kaggle with 2 GPUs, Cell 3 runs audio extraction, speaker diarization, transcription, and translation once (these stages aren't split). Then it launches **two GPU-pinned subprocesses** (`CUDA_VISIBLE_DEVICES=0` / `=1`), each synthesizing half of the dubbed voice lines in parallel, before merging the results and finishing background preservation, video muxing, and lip-sync. Every other entry point (Cell 2 / server mode, and the Colab-only single-GPU path) is unchanged and still uses one GPU.

In [ ]:
# Cell 1: Detect Environment (Colab / Kaggle) & Install Dependencies
import os
import subprocess

def detect_environment():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

def get_gpu_count():
    try:
        result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=15)
        return len([l for l in result.stdout.strip().split('\n') if l.strip().startswith('GPU')])
    except Exception:
        return 0

ENV = detect_environment()
PARENT_DIR = "/kaggle/working" if ENV == "kaggle" else "/content"
BASE_DIR = f"{PARENT_DIR}/dubbing_app"
GPU_COUNT = get_gpu_count()

print(f"Detected environment: {ENV.upper()}")
subprocess.run(['nvidia-smi'])

if ENV == "kaggle" and GPU_COUNT < 2:
    print(f"\nOnly {GPU_COUNT} GPU(s) visible on Kaggle. For the dual-T4 speedup in Cell 3, open "
          f"Notebook Settings (right sidebar) > Accelerator > GPU T4 x2, then restart the session and rerun this cell.")
elif ENV == "kaggle":
    print(f"\n{GPU_COUNT} GPUs detected on Kaggle - Cell 3 will automatically split voice "
          f"synthesis across both for a real speedup on the slowest stage.")
else:
    print(f"\n{GPU_COUNT} GPU(s) detected. Dual-GPU synthesis is Kaggle-only - Colab's free/Pro "
          f"tiers only ever expose a single GPU per session.")

# Always start from a clean known directory so re-running this cell (e.g. after a git pull)
# never accidentally "cd's into itself" and lands in a nested/wrong directory.
os.makedirs(PARENT_DIR, exist_ok=True)
os.chdir(PARENT_DIR)

if os.path.isdir(BASE_DIR):
    os.chdir(BASE_DIR)
    subprocess.run(['git', 'pull'])
else:
    subprocess.run(['git', 'clone', 'https://github.com/Asadullah404/Ai_Video_Dubbing.git', BASE_DIR])
    os.chdir(BASE_DIR)

print(f"\nWorking directory: {os.getcwd()}")

# Install server & dubbing dependencies cleanly without dependency conflict warnings
!pip install -q --no-warn-conflicts fastapi uvicorn python-multipart pycloudflared nest-asyncio deep-translator gdown ipywidgets
!pip install -q --no-warn-conflicts faster-whisper coqui-tts pyannote.audio audio-separator[gpu] speechbrain groq librosa soundfile noisereduce pedalboard resemblyzer gTTS yt-dlp opencv-python

# Install Chatterbox Multilingual voice cloning engine & prerequisites (--no-deps)
!pip install -q --no-warn-conflicts --no-deps chatterbox-tts diffusers==0.29.0 resemble-perth conformer==0.3.2 s3tokenizer pykakasi==2.3.0 spacy-pkuseg pyloudnorm

# Download pre-trained Wav2Lip and S3FD weights
!mkdir -p Wav2Lip/face_detection/detection/sfd
!wget -q -c 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights2/wav2lip_gan.1.1.pth' -O 'Wav2Lip/wav2lip_gan.pth'
!wget -q -c 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights1/s3fd-619a316812.1.1.pth' -O 'Wav2Lip/face_detection/detection/sfd/s3fd.pth'

print("\n✅ All Dependencies and Pre-trained Models are installed and ready!")
print(f"🚀 Environment: {ENV.upper()} | GPUs: {GPU_COUNT} | Base dir: {BASE_DIR}")
print("You can now directly run Cell 2 or Cell 3 below (no session restart required).")

In [ ]:
# Cell 2: Launch Cloud GPU Server & Expose via Cloudflare Tunnel
# Requires Cell 1 to have run first in this session (detects environment, installs deps, sets BASE_DIR).
# Note: server mode always uses a single GPU - the dual-T4 split is Cell 3 (one-click) only.
import os

# Optional: Set your HuggingFace token for PyAnnote speaker diarization
os.environ["HF_TOKEN"] = ""

# Groq API key (required for AI-powered translation) + up to 4 optional fallback keys.
# If the primary key hits Groq's rate limit mid-video, processing automatically shifts to
# the next key, then the next, and so on. Leave the fallback fields blank if you only have one key.
groq_api_key = ""       # required
groq_api_key_2 = ""     # optional fallback
groq_api_key_3 = ""     # optional fallback
groq_api_key_4 = ""     # optional fallback
groq_api_key_5 = ""     # optional fallback

os.environ["Groq_TOKEN"] = ",".join(
    k.strip() for k in [groq_api_key, groq_api_key_2, groq_api_key_3, groq_api_key_4, groq_api_key_5]
    if k.strip()
)
os.environ["COQUI_TOS_AGREED"] = "1"

print(f"Running server from: {os.getcwd()} (environment: {ENV.upper() if 'ENV' in dir() else 'unknown - run Cell 1 first'})")

# Run server
!python colab_server.py

In [ ]:
# Cell 3: One-Click Dubbing - interactive form (ipywidgets, works identically on Colab and Kaggle)
# Requires Cell 1 to have run first in this session.
import os
import sys
import json
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import display, Video, clear_output

# ============================================================================
# Environment + GPU detection (self-contained - safe even if Cell 1 wasn't rerun)
# ============================================================================

def detect_environment():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

def get_gpu_count():
    try:
        result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, timeout=15)
        return len([l for l in result.stdout.strip().split('\n') if l.strip().startswith('GPU')])
    except Exception:
        return 0

ENV = detect_environment()
BASE_DIR = "/kaggle/working/dubbing_app" if ENV == "kaggle" else "/content/dubbing_app"
GPU_COUNT = get_gpu_count()
USE_DUAL_GPU = (ENV == "kaggle" and GPU_COUNT >= 2)

os.chdir(BASE_DIR)

SOURCE_LANGS = ["auto", "en", "es", "fr", "de", "it", "pt", "pl", "tr", "ru", "nl", "cs", "ar",
                "zh-cn", "ja", "ko", "hi", "ur", "hu"]
TARGET_LANGS = ["en", "es", "fr", "de", "it", "pt", "pl", "tr", "ru", "nl", "cs", "ar", "zh-cn",
                "ja", "ko", "hi", "ur", "hu", "bn", "ta", "te", "ml", "th", "vi", "id", "ms",
                "fa", "sw", "ne", "si"]

WIDE = widgets.Layout(width="480px")

# ============================================================================
# Build the form widgets
# ============================================================================

video_source_w = widgets.Dropdown(
    options=["Upload from computer", "YouTube URL", "Google Drive Link"],
    value="Upload from computer", description="Video source:", style={"description_width": "140px"})
youtube_url_w = widgets.Text(description="YouTube URL:", placeholder="https://www.youtube.com/watch?v=...",
                              layout=WIDE, style={"description_width": "140px"})
youtube_cookies_w = widgets.Text(description="YT cookies path:", placeholder="/path/to/cookies.txt",
                                  layout=WIDE, style={"description_width": "140px"})
gdrive_link_w = widgets.Text(description="Drive share link:", placeholder="https://drive.google.com/file/d/.../view",
                              layout=WIDE, style={"description_width": "140px"})

source_lang_w = widgets.Dropdown(options=SOURCE_LANGS, value="en", description="Source language:",
                                  style={"description_width": "140px"})
target_lang_w = widgets.Dropdown(options=TARGET_LANGS, value="es", description="Target language:",
                                  style={"description_width": "140px"})
whisper_model_w = widgets.Dropdown(options=["tiny", "base", "small", "medium", "large-v3"], value="large-v3",
                                    description="Whisper model:", style={"description_width": "140px"})
voice_quality_w = widgets.Dropdown(options=["standard", "high", "ultra"], value="ultra",
                                    description="Voice quality:", style={"description_width": "140px"})

enable_lipsync_w = widgets.Checkbox(value=True, description="Enable lip-sync")
preserve_bg_w = widgets.Checkbox(value=True, description="Preserve background audio")

hf_token_w = widgets.Password(description="HF token:", layout=WIDE, style={"description_width": "140px"})
groq_key_w = widgets.Password(description="Groq key (required):", layout=WIDE, style={"description_width": "140px"})
groq_key2_w = widgets.Password(description="Groq fallback 2:", layout=WIDE, style={"description_width": "140px"})
groq_key3_w = widgets.Password(description="Groq fallback 3:", layout=WIDE, style={"description_width": "140px"})
groq_key4_w = widgets.Password(description="Groq fallback 4:", layout=WIDE, style={"description_width": "140px"})
groq_key5_w = widgets.Password(description="Groq fallback 5:", layout=WIDE, style={"description_width": "140px"})

start_btn = widgets.Button(description="Start Processing", button_style="success", icon="play")
out = widgets.Output()

def update_visibility(*_args):
    src = video_source_w.value
    youtube_url_w.layout.display = "" if src == "YouTube URL" else "none"
    youtube_cookies_w.layout.display = "" if src == "YouTube URL" else "none"
    gdrive_link_w.layout.display = "" if src == "Google Drive Link" else "none"

video_source_w.observe(update_visibility, names="value")
update_visibility()

upload_note_html = (
    "<i>Note: 'Upload from computer' only works on Colab (Kaggle has no in-cell upload widget) - "
    "use YouTube URL or Google Drive Link instead.</i>" if ENV != "colab" else ""
)
info_label = widgets.HTML(
    f"<b>Environment:</b> {ENV.upper()} &nbsp;|&nbsp; <b>GPUs:</b> {GPU_COUNT} &nbsp;|&nbsp; "
    f"<b>Dual-GPU synthesis:</b> {'ON' if USE_DUAL_GPU else 'off'}<br>{upload_note_html}"
)

form = widgets.VBox([
    info_label,
    video_source_w, youtube_url_w, youtube_cookies_w, gdrive_link_w,
    widgets.HBox([source_lang_w, target_lang_w]),
    widgets.HBox([whisper_model_w, voice_quality_w]),
    widgets.HBox([enable_lipsync_w, preserve_bg_w]),
    widgets.HTML("<b>API Keys</b> (Groq: 1 required + up to 4 optional fallback keys - auto fail-over on rate limit)"),
    hf_token_w, groq_key_w, groq_key2_w, groq_key3_w, groq_key4_w, groq_key5_w,
    start_btn,
])
display(form, out)

# ============================================================================
# Subprocess launch + live streaming helpers
# ============================================================================

def run_and_stream(cmd, env=None, prefix=""):
    """Runs cmd, streams stdout live (prefixed), returns the exit code."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(f"{prefix}{line}", end="")
    proc.wait()
    return proc.returncode

def run_parallel_and_stream(cmds_envs_prefixes):
    """Runs multiple (cmd, env, prefix) tuples concurrently, streaming all of their output live
    (each line tagged with its own prefix so the two GPUs' logs stay distinguishable), and
    returns a list of exit codes in the same order."""
    procs = []
    for cmd, env, prefix in cmds_envs_prefixes:
        procs.append((subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                        text=True, bufsize=1, env=env), prefix))

    print_lock = threading.Lock()

    def reader(proc, prefix):
        for line in proc.stdout:
            with print_lock:
                print(f"{prefix}{line}", end="")

    threads = [threading.Thread(target=reader, args=(p, prefix)) for p, prefix in procs]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    for p, _ in procs:
        p.wait()
    return [p.returncode for p, _ in procs]

# ============================================================================
# Pipeline runner - triggered by the Start Processing button
# ============================================================================

def run_pipeline(_btn):
    start_btn.disabled = True
    start_btn.description = "Processing..."
    out.clear_output()
    try:
        with out:
            _run_pipeline_body()
    finally:
        start_btn.disabled = False
        start_btn.description = "Start Processing"

def _run_pipeline_body():
    video_source = video_source_w.value
    youtube_url = youtube_url_w.value
    youtube_cookies_path = youtube_cookies_w.value
    gdrive_link = gdrive_link_w.value
    source_language = source_lang_w.value
    target_language = target_lang_w.value
    whisper_model = whisper_model_w.value
    voice_quality = voice_quality_w.value
    enable_lipsync = enable_lipsync_w.value
    preserve_background_audio = preserve_bg_w.value
    hf_token = hf_token_w.value
    groq_api_key = groq_key_w.value
    groq_api_key_2 = groq_key2_w.value
    groq_api_key_3 = groq_key3_w.value
    groq_api_key_4 = groq_key4_w.value
    groq_api_key_5 = groq_key5_w.value

    os.chdir(BASE_DIR)
    print(f"Environment: {ENV.upper()} | GPUs: {GPU_COUNT} | Dual-GPU synthesis: {'ON' if USE_DUAL_GPU else 'off'}")

    if hf_token.strip():
        os.environ["HF_TOKEN"] = hf_token.strip()

    groq_keys = [k.strip() for k in [groq_api_key, groq_api_key_2, groq_api_key_3, groq_api_key_4, groq_api_key_5] if k.strip()]
    if groq_keys:
        os.environ["Groq_TOKEN"] = ",".join(groq_keys)
        if len(groq_keys) > 1:
            print(f"Groq: {len(groq_keys)} API keys loaded (auto fail-over on rate limit)")

    if youtube_cookies_path.strip():
        os.environ["YT_COOKIES_FILE"] = youtube_cookies_path.strip()
    os.environ["COQUI_TOS_AGREED"] = "1"

    # ------------------------------------------------------------------
    # Resolve the video source
    # ------------------------------------------------------------------
    uploaded_video_path = ""

    if video_source == "Upload from computer":
        if ENV != "colab":
            raise Exception("Direct upload only works on Colab (files.upload() isn't available on "
                             "Kaggle). On Kaggle, use 'Google Drive Link' or 'YouTube URL' instead - "
                             "both work identically here.")
        from google.colab import files
        print("Choose a video file to upload...")
        uploaded = files.upload()
        if not uploaded:
            raise Exception("No file uploaded.")
        uploaded_video_path = list(uploaded.keys())[0]
        print(f"\nUploaded video ready: {uploaded_video_path}")

    elif video_source == "Google Drive Link":
        uploaded_video_path = "gdrive_input_video.mp4"
        if os.path.exists(uploaded_video_path):
            print(f"Reusing already-downloaded video: {uploaded_video_path}")
        else:
            if not gdrive_link.strip():
                raise Exception("Please paste a Google Drive share link in the form above.")
            try:
                import gdown
            except ImportError:
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
                import gdown
            print("Downloading video from Google Drive...")
            uploaded_video_path = gdown.download(url=gdrive_link.strip(), output=uploaded_video_path,
                                                  quiet=False, fuzzy=True)
            if not uploaded_video_path or not os.path.exists(uploaded_video_path):
                raise Exception("Google Drive download failed - make sure the file's sharing is set "
                                 "to 'Anyone with the link'.")
            print(f"\nDownloaded video ready: {uploaded_video_path}")

    else:
        if not youtube_url.strip():
            raise Exception("Please paste a YouTube URL in the form above.")

    # ------------------------------------------------------------------
    # Run the pipeline
    # ------------------------------------------------------------------

    if not USE_DUAL_GPU:
        # Single-GPU path (Colab, or Kaggle with only 1 GPU selected): run the
        # whole pipeline in one clean subprocess.
        runner_code = f"""
import os
from video_dubbing_core import EnhancedVideoDubbing, download_youtube_video

video_source = {video_source!r}
youtube_url = {youtube_url.strip()!r}
video_path = {uploaded_video_path!r}

if video_source == "YouTube URL":
    print("Downloading YouTube video...")
    video_path = download_youtube_video(youtube_url)
    if not video_path:
        print("YouTube download failed - check the URL or cookies.")
        raise SystemExit(1)

print(f"\\nStarting AI Video Dubbing pipeline for: {{video_path}}")

dubber = EnhancedVideoDubbing(
    video_path=video_path,
    source_lang={source_language!r},
    target_lang={target_language!r},
    whisper_model={whisper_model!r},
    voice_quality={voice_quality!r},
    enable_lipsync={enable_lipsync!r},
    preserve_bg={preserve_background_audio!r},
    hf_token=os.getenv("HF_TOKEN"),
    groq_token=os.getenv("Groq_TOKEN"),
    reset_progress=False,
)
dubber.process()
"""
        returncode = run_and_stream([sys.executable, "-u", "-c", runner_code])
        if returncode != 0:
            raise RuntimeError(f"Processing failed (exit code {returncode}) - check log above.")

    else:
        # Dual-GPU path (Kaggle, 2x T4): run stages 1-4 once, split voice
        # synthesis across both GPUs in parallel, then finish stages 6-9.
        print("\n=== Stage 1-4: audio extraction, diarization, transcription, translation ===")
        stage1_4_code = f"""
import os, json
from video_dubbing_core import EnhancedVideoDubbing, download_youtube_video

video_source = {video_source!r}
youtube_url = {youtube_url.strip()!r}
video_path = {uploaded_video_path!r}

if video_source == "YouTube URL":
    print("Downloading YouTube video...")
    video_path = download_youtube_video(youtube_url)
    if not video_path:
        print("YouTube download failed - check the URL or cookies.")
        raise SystemExit(1)

print(f"\\nStarting AI Video Dubbing pipeline for: {{video_path}}")

dubber = EnhancedVideoDubbing(
    video_path=video_path,
    source_lang={source_language!r},
    target_lang={target_language!r},
    whisper_model={whisper_model!r},
    voice_quality={voice_quality!r},
    enable_lipsync={enable_lipsync!r},
    preserve_bg={preserve_background_audio!r},
    hf_token=os.getenv("HF_TOKEN"),
    groq_token=os.getenv("Groq_TOKEN"),
    reset_progress=False,
)
dubber.process(stop_before_synthesis=True)
json.dump({{"video_path": video_path}}, open("_v2_resolved_video.json", "w"))
"""
        returncode = run_and_stream([sys.executable, "-u", "-c", stage1_4_code])
        if returncode != 0:
            raise RuntimeError(f"Stage 1-4 failed (exit code {returncode}) - check log above.")

        resolved = json.load(open("_v2_resolved_video.json"))
        resolved_video_path = resolved["video_path"]

        print(f"\n=== Stage 5: voice synthesis - splitting across {GPU_COUNT} GPUs ===")
        shard_template = """
import os, sys, json
os.chdir({base_dir!r})

shard_index = {shard_index}
num_shards = {num_shards}
video_path = {video_path!r}
progress_file = f"processing_progress_shard{{shard_index}}.json"

# Pre-seed this shard's own progress file with the matching video_path and a non-'complete'
# stage - otherwise EnhancedVideoDubbing.__init__ treats it as an unrelated/new video and
# calls ProgressTracker.reset(), which deletes the shared checkpoint_*.json files this shard
# (and the other shard) are about to read from.
if not os.path.exists(progress_file):
    json.dump({{"stage": "translation_done", "completed_chunks": [], "completed_segments": [],
                "timestamp": "", "video_path": video_path}}, open(progress_file, "w"))

from video_dubbing_core import EnhancedVideoDubbing

dubber = EnhancedVideoDubbing(
    video_path=video_path,
    source_lang={source_language!r},
    target_lang={target_language!r},
    whisper_model={whisper_model!r},
    voice_quality={voice_quality!r},
    enable_lipsync={enable_lipsync!r},
    preserve_bg={preserve_background_audio!r},
    hf_token=os.getenv("HF_TOKEN"),
    groq_token=os.getenv("Groq_TOKEN"),
    reset_progress=False,
    progress_file=progress_file,
)

records = dubber.load_checkpoint('translated_records')
raw_rolls = dubber.load_checkpoint('speakers_rolls') or {{}}
speakers_rolls = {{tuple(map(float, k.split('_'))): v for k, v in raw_rolls.items()}}

my_indices = {{i for i in range(len(records)) if i % num_shards == shard_index}}
print(f"[GPU {{shard_index}}] synthesizing {{len(my_indices)}} of {{len(records)}} segments")
dubber.synthesize_with_timing(records, speakers_rolls, segment_indices=my_indices)
print(f"[GPU {{shard_index}}] shard finished")
"""

        shard_jobs = []
        for shard_index in range(2):
            code = shard_template.format(
                base_dir=BASE_DIR, shard_index=shard_index, num_shards=2,
                video_path=resolved_video_path, source_language=source_language,
                target_language=target_language, whisper_model=whisper_model,
                voice_quality=voice_quality, enable_lipsync=enable_lipsync,
                preserve_background_audio=preserve_background_audio,
            )
            shard_env = os.environ.copy()
            shard_env["CUDA_VISIBLE_DEVICES"] = str(shard_index)
            shard_jobs.append(([sys.executable, "-u", "-c", code], shard_env, f"[GPU{shard_index}] "))

        shard_returncodes = run_parallel_and_stream(shard_jobs)
        if any(rc != 0 for rc in shard_returncodes):
            raise RuntimeError(f"A GPU synthesis shard failed (exit codes {shard_returncodes}) - check log above.")

        # Merge both shards' completed-segment lists into the main progress file, and fast-forward
        # its stage past synthesis only if every segment truly finished - otherwise leave the stage
        # as-is so the next step's normal synthesize_with_timing() call catches any stragglers.
        from video_dubbing_core import ProgressTracker
        main_progress = ProgressTracker("processing_progress.json")
        total_segments = len(json.load(open("checkpoint_translated_records.json")))
        for shard_index in range(2):
            shard_file = f"processing_progress_shard{shard_index}.json"
            if os.path.exists(shard_file):
                shard_data = json.load(open(shard_file))
                for seg in shard_data.get("completed_segments", []):
                    main_progress.mark_segment_done(seg)
        if len(main_progress.progress.get("completed_segments", [])) >= total_segments:
            main_progress.update_stage("synthesis_done")
            print(f"\nAll {total_segments} segments synthesized across both GPUs.")
        else:
            done = len(main_progress.progress.get("completed_segments", []))
            print(f"\n{done}/{total_segments} segments synthesized - remaining will be filled in next.")

        print("\n=== Stage 6-9: assembly, background preservation, video muxing, lip-sync ===")
        continuation_code = f"""
import os
from video_dubbing_core import EnhancedVideoDubbing

dubber = EnhancedVideoDubbing(
    video_path={resolved_video_path!r},
    source_lang={source_language!r},
    target_lang={target_language!r},
    whisper_model={whisper_model!r},
    voice_quality={voice_quality!r},
    enable_lipsync={enable_lipsync!r},
    preserve_bg={preserve_background_audio!r},
    hf_token=os.getenv("HF_TOKEN"),
    groq_token=os.getenv("Groq_TOKEN"),
    reset_progress=False,
)
dubber.process()
"""
        returncode = run_and_stream([sys.executable, "-u", "-c", continuation_code])
        if returncode != 0:
            raise RuntimeError(f"Stage 6-9 failed (exit code {returncode}) - check log above.")

    # ------------------------------------------------------------------
    # Show and download the result
    # ------------------------------------------------------------------
    result_path = "results/dubbed_video.mp4"
    if os.path.exists(result_path):
        print(f"\nDone! Output: {result_path}")
        display(Video(result_path, embed=True, width=640))
        if ENV == "colab":
            from google.colab import files
            files.download(result_path)
        else:
            print(f"Video saved to: {os.path.abspath(result_path)} (download it from the Kaggle Output/Files panel)")
    else:
        print("Processing finished but the output video wasn't found - check the log above for errors.")

start_btn.on_click(run_pipeline)